In [1]:
import os
import h5py
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from tqdm import tqdm
import logging
import csv
import numpy as np

In [2]:
class MagNetH5Dataset(Dataset):
    def __init__(self, h5_path, normalize=True):
        self.h5_path = h5_path
        self.normalize = normalize
        
        with h5py.File(h5_path, 'r') as f:
            self.length = len(f['P'])
            self.materials = np.unique(f['material'][:]).tolist()
            self.mat_to_idx = {m.decode('utf-8'): i for i, m in enumerate(self.materials)}
            
            if normalize:
                self.B_mean = float(f['B'][:].mean())
                self.B_std  = float(f['B'][:].std()) + 1e-8
                self.f_mean = float(f['f'][:].mean())
                self.f_std  = float(f['f'][:].std()) + 1e-8
                self.T_mean = float(f['T'][:].mean())
                self.T_std  = float(f['T'][:].std()) + 1e-8
                self.P_log_mean = float(np.log1p(f['P'][:]).mean())
                self.P_log_std  = float(np.log1p(f['P'][:]).std()) + 1e-8

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        with h5py.File(self.h5_path, 'r') as f:
            B = torch.from_numpy(f['B'][idx].astype(np.float32)).unsqueeze(0)
            f_val = torch.tensor([f['f'][idx]], dtype=torch.float32)
            T_val = torch.tensor([f['T'][idx]], dtype=torch.float32)
            P_raw = torch.tensor([f['P'][idx]], dtype=torch.float32)
            
            mat_name = f['material'][idx].decode('utf-8')
            mat_onehot = F.one_hot(torch.tensor(self.mat_to_idx[mat_name]), num_classes=10).float()
            tabular = torch.cat([f_val, T_val, mat_onehot]) # Total 12 dims

            if self.normalize:
                B = (B - self.B_mean) / self.B_std
                tabular[0] = (tabular[0] - self.f_mean) / self.f_std
                tabular[1] = (tabular[1] - self.T_mean) / self.T_std
                P = torch.log1p(P_raw)
                P = (P - self.P_log_mean) / self.P_log_std
            else:
                P = P_raw

        return B, tabular, P

In [3]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x):
        seq_len = x.size(1)
        return x + self.pe[:seq_len, :].unsqueeze(0)

class MultimodalInputEncoder(nn.Module):
    def __init__(self, tabular_dim=12, embed_dim=256):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(1, 64, 7, padding=3), nn.GELU(), nn.MaxPool1d(2),
            nn.Conv1d(64, 128, 7, padding=3), nn.GELU(), nn.MaxPool1d(2),
            nn.Conv1d(128, 128, 7, padding=3), nn.GELU()
        )
        self.wave_proj = nn.Linear(128 + 1, embed_dim)
        
        self.tab_mlp = nn.Sequential(
            nn.Linear(tabular_dim, 256), 
            nn.GELU(), 
            nn.BatchNorm1d(256), 
            nn.Linear(256, embed_dim)
        )
        self.pos_encoder = PositionalEncoding(embed_dim, max_len=10)
        self.cross_attn = nn.MultiheadAttention(embed_dim, num_heads=8, batch_first=True)

    def forward(self, waveform, tabular):
        cnn_feat = self.cnn(waveform)
        seq_len_out = cnn_feat.shape[-1]
        
        fft_complex = torch.fft.rfft(waveform, dim=-1)
        fft_mag = torch.abs(fft_complex)
        fft_mag_pooled = F.adaptive_avg_pool1d(fft_mag, seq_len_out)
        
        wave_concat = torch.cat([cnn_feat, fft_mag_pooled], dim=1).transpose(1, 2)
        query = self.wave_proj(wave_concat)
        
        tab_feat = self.tab_mlp(tabular).unsqueeze(1)
        key_value = self.pos_encoder(tab_feat)
        
        fused, _ = self.cross_attn(query=query, key=key_value, value=key_value)
        return fused

In [4]:
class PhysicsInformedResidualLayer(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.bottleneck = nn.Linear(d_model, 5)
        self.residual_proj = nn.Linear(2, d_model)

    def forward(self, x, waveform, tabular):
        phys = F.softplus(self.bottleneck(x))
        k, alpha, beta, A, Ea = phys.unbind(-1)
        B_max = torch.max(torch.abs(waveform), dim=-1)[0].squeeze(1) + 1e-4
        f = F.softplus(tabular[:, 0]) + 1e-4
        T = F.softplus(tabular[:, 1]) + 1.0
        
        A = torch.clamp(A, min=1e-3, max=50.0)
        Ea = torch.clamp(Ea, min=0.1, max=15.0)
        arrhenius = torch.clamp(A * torch.exp(-Ea / T), max=500.0)
        steinmetz = torch.clamp(k * (f ** alpha) * (B_max ** beta), max=500.0)
        
        residuals = torch.stack([steinmetz, arrhenius], dim=-1)
        residuals = torch.clamp(residuals, -200.0, 200.0)
        return x + self.residual_proj(residuals)

In [5]:
class TemporalBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel=3, dilation=1, dropout=0.1):
        super().__init__()
        self.padding = (kernel - 1) * dilation
        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel, padding=self.padding, dilation=dilation)
        self.norm1 = nn.GroupNorm(8, out_ch)
        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel, padding=self.padding, dilation=dilation)
        self.norm2 = nn.GroupNorm(8, out_ch)
        self.drop = nn.Dropout(dropout)
        self.down = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else None
    def forward(self, x):
        out = self.conv1(x)
        if self.padding > 0: out = out[:, :, :-self.padding]
        out = self.drop(F.gelu(self.norm1(out)))
        out = self.conv2(out)
        if self.padding > 0: out = out[:, :, :-self.padding]
        out = self.drop(F.gelu(self.norm2(out)))
        res = x if self.down is None else self.down(x)
        return F.gelu(out + res)

class TCNBackbone(nn.Module):
    def __init__(self, d_model=256, num_layers=4):
        super().__init__()
        layers = [TemporalBlock(d_model if i == 0 else d_model, d_model, dilation=2**i) for i in range(num_layers)]
        self.network = nn.Sequential(*layers)
    def forward(self, x):
        return self.network(x.transpose(1, 2)).transpose(1, 2)

class LSTMBackbone(nn.Module):
    def __init__(self, d_model=256, num_layers=2, bidirectional=False):
        super().__init__()
        hidden_size = d_model // 2 if bidirectional else d_model
        self.lstm = nn.LSTM(d_model, hidden_size, num_layers, batch_first=True, bidirectional=bidirectional)
    def forward(self, x):
        x, _ = self.lstm(x)
        return x

class GRUBackbone(nn.Module):
    def __init__(self, d_model=256, num_layers=2, bidirectional=False):
        super().__init__()
        hidden_size = d_model // 2 if bidirectional else d_model
        self.gru = nn.GRU(d_model, hidden_size, num_layers, batch_first=True, bidirectional=bidirectional)
    def forward(self, x):
        x, _ = self.gru(x)
        return x

class LSTMAttention(nn.Module):
    def __init__(self, d_model=256, num_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(d_model, d_model//2, num_layers, batch_first=True, bidirectional=True)
        self.attn = nn.Sequential(nn.Linear(d_model, d_model//2), nn.Tanh(), nn.Linear(d_model//2, 1))
    def forward(self, x):
        x, _ = self.lstm(x)
        scores = self.attn(x).squeeze(-1)
        weights = F.softmax(scores, dim=1).unsqueeze(-1)
        return (weights * x).sum(dim=1).unsqueeze(1)

class RWKVTimeMixing(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.time_shift = nn.ZeroPad2d((0, 0, 1, -1))
        self.key = nn.Linear(d_model, d_model, bias=False)
        self.value = nn.Linear(d_model, d_model, bias=False)
        self.receptance = nn.Linear(d_model, d_model, bias=False)
        self.output = nn.Linear(d_model, d_model, bias=False)
        self.time_mix_k = nn.Parameter(torch.ones(1, 1, d_model))
        self.time_mix_v = nn.Parameter(torch.ones(1, 1, d_model))
        self.time_mix_r = nn.Parameter(torch.ones(1, 1, d_model))
        self.time_decay = nn.Parameter(torch.ones(d_model))
        self.time_first = nn.Parameter(torch.ones(d_model))

    def forward(self, x):
        B, T, C = x.shape
        x_prev = self.time_shift(x)
        k = self.key(x * self.time_mix_k + x_prev * (1 - self.time_mix_k))
        v = self.value(x * self.time_mix_v + x_prev * (1 - self.time_mix_v))
        r = self.receptance(x * self.time_mix_r + x_prev * (1 - self.time_mix_r))

        wkv = torch.zeros_like(v)
        a = torch.zeros((B, C), device=x.device)
        b = torch.zeros((B, C), device=x.device)
        p = torch.full((B, C), -1e38, device=x.device)
        for t in range(T):
            kt = k[:, t, :]
            vt = v[:, t, :]
            q = torch.maximum(p, self.time_first + kt)
            e1 = torch.exp(torch.clamp(p - q, min=-30.0, max=0.0))
            e2 = torch.exp(torch.clamp(self.time_first + kt - q, min=-30.0, max=0.0))
            wkv[:, t, :] = (e1 * a + e2 * vt) / (e1 * b + e2 + 1e-8)
            q_new = torch.maximum(p - self.time_decay, kt)
            e1_new = torch.exp(torch.clamp(p - self.time_decay - q_new, min=-30.0, max=0.0))
            e2_new = torch.exp(torch.clamp(kt - q_new, min=-30.0, max=0.0))
            a = e1_new * a + e2_new * vt
            b = e1_new * b + e2_new
            p = q_new
        return self.output(torch.sigmoid(r) * wkv)

class RWKVBlock(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.time_mix = RWKVTimeMixing(d_model)
        self.channel_mix = nn.Sequential(nn.Linear(d_model, d_model*4), nn.GELU(), nn.Linear(d_model*4, d_model))
    def forward(self, x):
        x = x + self.time_mix(self.ln1(x))
        x = x + self.channel_mix(self.ln2(x))
        return x

class xLSTMsBlock(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.s_lstm = nn.LSTM(d_model, d_model, 1, batch_first=True)
        self.channel = nn.Sequential(nn.Linear(d_model, d_model*4), nn.GELU(), nn.Linear(d_model*4, d_model))
    def forward(self, x):
        x = x + self.s_lstm(self.ln1(x))[0]
        x = x + self.channel(self.ln2(x))
        return x

In [6]:
class MagNetCoreLossModel(nn.Module):
    def __init__(self, variant="xLSTM", d_model=256, num_layers=4):
        super().__init__()
        self.variant = variant
        self.encoder = MultimodalInputEncoder(tabular_dim=12, embed_dim=d_model)
        
        if variant == "TCN":
            self.backbone = TCNBackbone(d_model, num_layers)
        elif variant == "LSTM":
            self.backbone = LSTMBackbone(d_model, num_layers, bidirectional=False)
        elif variant == "BiLSTM":
            self.backbone = LSTMBackbone(d_model, num_layers, bidirectional=True)
        elif variant == "GRU":
            self.backbone = GRUBackbone(d_model, num_layers, bidirectional=False)
        elif variant == "BiGRU":
            self.backbone = GRUBackbone(d_model, num_layers, bidirectional=True)
        elif variant == "LSTM-Attention":
            self.backbone = LSTMAttention(d_model, num_layers)
        elif variant == "RWKV":
            self.backbone = nn.Sequential(*[RWKVBlock(d_model) for _ in range(num_layers)])
        elif variant == "xLSTM":
            self.backbone = nn.Sequential(*[xLSTMsBlock(d_model) for _ in range(num_layers)])
        else:
            raise ValueError(f"Unknown variant: {variant}")
        
        self.physics = PhysicsInformedResidualLayer(d_model)
        self.norm = nn.LayerNorm(d_model)
        
        self.core_head = nn.Sequential(nn.Linear(d_model, 64), nn.GELU(), nn.Linear(64, 1))
        self.rul_head = nn.Sequential(nn.Linear(d_model, 64), nn.GELU(), nn.Linear(64, 2))
    
    def forward(self, waveform, tabular):
        x = self.encoder(waveform, tabular)
        x = self.backbone(x)
        x_phys = self.physics(x[:, -1, :], waveform, tabular) 
        x_out = self.norm(x_phys)
        
        core_pred = self.core_head(x_out).squeeze(-1)
        rul_pred = self.rul_head(x_out)
        return core_pred, rul_pred

In [7]:
def calculate_metrics(pred, target):
    pred = torch.clamp(torch.nan_to_num(pred, 0.0), -10, 10)
    target = torch.nan_to_num(target, 0.0)
    mae = F.l1_loss(pred, target).item()
    rmse = torch.sqrt(F.mse_loss(pred, target)).item()
    ss_res = ((target - pred) ** 2).sum()
    ss_tot = ((target - target.mean()) ** 2).sum()
    r2 = float(torch.clamp(1 - ss_res / ss_tot, -100, 1)) if ss_tot > 1e-8 else 0.0
    rel_err = torch.abs((target - pred) / (torch.abs(target) + 1e-8)) * 100
    p95 = torch.quantile(rel_err, 0.95).item()
    return mae, rmse, r2, p95

def pretrain_variant(variant_name="xLSTM", epochs=10, batch_size=48, lr=3e-5):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    checkpoint_dir = f"checkpoints_mepi_pretrain_done/{variant_name}"
    os.makedirs(checkpoint_dir, exist_ok=True)
    
    log_file = f"{checkpoint_dir}/training_log.txt"
    logger = logging.getLogger(variant_name)
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    fh = logging.FileHandler(log_file, encoding='utf-8')
    ch = logging.StreamHandler()
    formatter = logging.Formatter('%(asctime)s - %(message)s')
    fh.setFormatter(formatter)
    ch.setFormatter(formatter)
    logger.addHandler(fh)
    logger.addHandler(ch)
    
    csv_path = f"{checkpoint_dir}/metrics.csv"
    csv_file = open(csv_path, 'w', newline='', encoding='utf-8')
    writer = csv.writer(csv_file)
    writer.writerow(["Step", "Loss", "MAE", "RMSE", "R2", "95th_Pct_Error", "GradNorm"])
    
    logger.info(f"🚀 BẮT ĐẦU PRETRAIN (MEPI FULL) {variant_name} | Device: {device} | Batch: {batch_size} | LR: {lr}")
    
    dataset = MagNetH5Dataset("magnet_pretrain_186k.h5", normalize=True)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=12, pin_memory=True, drop_last=True)
    
    model = MagNetCoreLossModel(variant=variant_name).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scaler = GradScaler(enabled=torch.cuda.is_available())
    
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr, total_steps=len(loader)*epochs,
        pct_start=0.1, anneal_strategy='cos', div_factor=25.0
    )
    
    best_mae = float('inf')
    best_path = f"{checkpoint_dir}/best.pth"
    global_step = 0
    
    for epoch in range(epochs):
        model.train()
        pbar = tqdm(loader, desc=f"Epoch {epoch+1}/{epochs} [{variant_name}]")
        
        for B, tabular, P in pbar:
            B = B.to(device, non_blocking=True)
            tabular = tabular.to(device, non_blocking=True)
            P = P.to(device, non_blocking=True).view(-1)
            
            optimizer.zero_grad()
            with autocast(device_type='cuda', enabled=torch.cuda.is_available()):
        
                core_pred, _ = model(B, tabular)
                loss = F.mse_loss(core_pred, P) + 0.3 * F.l1_loss(core_pred, P)
            
            scaler.scale(loss).backward()
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            
            global_step += 1
            
            if global_step % 100 == 0:
                mae, rmse, r2, p95 = calculate_metrics(core_pred, P)
                logger.info(f"Step {global_step:5d} | Loss: {loss.item():.4f} | "
                           f"MAE: {mae:.4f} | RMSE: {rmse:.4f} | R²: {r2:.4f} | "
                           f"95th: {p95:.2f}% | GradNorm: {grad_norm:.2f}")
                writer.writerow([global_step, loss.item(), mae, rmse, r2, p95, grad_norm])
                csv_file.flush()
                
                if mae < best_mae:
                    best_mae = mae
                    torch.save(model.state_dict(), best_path)
                    logger.info(f"🏆 BEST MODEL UPDATED! MAE = {best_mae:.4f} (saved to {best_path})")
    
    csv_file.close()
    logger.info(f"HOÀN THÀNH {variant_name} | Best MAE = {best_mae:.4f}")
    return best_mae

In [8]:
if __name__ == "__main__":
    variants = ["TCN", "LSTM", "BiLSTM", "LSTM-Attention", "GRU", "BiGRU", "RWKV", "xLSTM"]
    
    results = {}
    for v in variants:
        try:
            mae = pretrain_variant(variant_name=v, epochs=20, batch_size=64, lr=3e-5)
            results[v] = mae
        except Exception as e:
            print(f"Lỗi khi pre-train {v}: {e}")
    
    print("\n=== KẾT QUẢ SO SÁNH PRE-TRAINING ===")
    for v, mae in sorted(results.items(), key=lambda x: x[1]):
        print(f"{v:15} → Best MAE = {mae:.6f}")

2026-05-02 02:18:51,864 - 🚀 BẮT ĐẦU PRETRAIN (MEPI FULL) TCN | Device: cuda | Batch: 64 | LR: 3e-05
Epoch 1/20 [TCN]:   3%|▎         | 99/2918 [00:07<02:38, 17.80it/s] 2026-05-02 02:19:10,684 - Step   100 | Loss: 1.3682 | MAE: 0.8914 | RMSE: 1.0492 | R²: 0.0095 | 95th: 184.42% | GradNorm: 523614.97
2026-05-02 02:19:10,756 - 🏆 BEST MODEL UPDATED! MAE = 0.8914 (saved to checkpoints_mepi_pretrain_done/TCN/best.pth)
Epoch 1/20 [TCN]:   7%|▋         | 199/2918 [00:13<02:26, 18.53it/s]2026-05-02 02:19:16,211 - Step   200 | Loss: 1.2021 | MAE: 0.7955 | RMSE: 0.9816 | R²: 0.0581 | 95th: 150.46% | GradNorm: 259448.41
2026-05-02 02:19:16,236 - 🏆 BEST MODEL UPDATED! MAE = 0.7955 (saved to checkpoints_mepi_pretrain_done/TCN/best.pth)
Epoch 1/20 [TCN]:  17%|█▋        | 498/2918 [00:29<02:17, 17.59it/s]2026-05-02 02:19:32,217 - Step   500 | Loss: 0.8207 | MAE: 0.6362 | RMSE: 0.7936 | R²: 0.3838 | 95th: 601.46% | GradNorm: 201240.92
2026-05-02 02:19:32,239 - 🏆 BEST MODEL UPDATED! MAE = 0.6362 (saved 


=== KẾT QUẢ SO SÁNH PRE-TRAINING ===
TCN             → Best MAE = 0.068865
LSTM            → Best MAE = 0.069164
BiGRU           → Best MAE = 0.069417
xLSTM           → Best MAE = 0.071595
BiLSTM          → Best MAE = 0.071617
RWKV            → Best MAE = 0.072071
LSTM-Attention  → Best MAE = 0.075489
GRU             → Best MAE = 0.078731
